# LangGraph Research Loop — Unconstrained Cost Runup

## The Problem

LangGraph's power is conditional edges: the LLM decides when the graph moves to the
next node. A research agent that loops through `search → analyze → search` is correct
by design. The LLM keeps searching until it's satisfied. On a clear query it stops at
5 iterations. On an ambiguous one it runs 40+. **Same graph, 8× the cost.**

You can add `max_iterations=20` but that's a count ceiling, not a cost ceiling.
A session with 20 cheap searches costs the same as one with 5 expensive ones —
the count guard doesn't help you when per-call cost varies
(Serper vs Perplexity vs a paid financial data API).

## The Fix

FiGuard wraps the search tool. Each call authorizes the per-call cost before it fires.
When the session budget is exhausted the authorization is denied, the tool raises
`BudgetExhausted`, and the graph exits the loop with whatever it has already gathered.
Cost is bounded by dollars, not by iteration count.

In [ ]:
!pip install figuard langgraph langchain langchain-openai -q

In [ ]:
# ── Configuration ──────────────────────────────────────────────────────────────
MODE = "simulation"  # Change to "real" to use your own API keys

if MODE == "real":
    from google.colab import userdata
    OPENAI_API_KEY = userdata.get("OPENAI_API_KEY")   # Add via Colab Secrets panel (🔑)
    STRIPE_TEST_KEY = userdata.get("STRIPE_TEST_KEY") # Only needed for langchain scenario
else:
    OPENAI_API_KEY = None
    STRIPE_TEST_KEY = None

FIGUARD_BASE_URL = "https://figuard-sandbox-g1ha.onrender.com"
FIGUARD_API_KEY  = "sb_live_demo"

# Note: FiGuard always runs against the live sandbox — real authorization
# decisions and spend tree — even in simulation mode.

In [ ]:
# ── Display helpers ────────────────────────────────────────────────────────────
def section(title):
    print(f"\n{'═' * 64}")
    print(f"  {title}")
    print(f"{'═' * 64}")

def ok(msg):   print(f"  ✓  {msg}")
def bad(msg):  print(f"  ✗  {msg}")
def info(msg): print(f"     {msg}")
def step(msg): print(f"  →  {msg}")

In [ ]:
# ── Scenario tuning ────────────────────────────────────────────────────────────
# How many search iterations the simulated agent will want to run.
SIMULATED_ITERATIONS = 30

# Cost FiGuard charges per search call (in USD).
COST_PER_SEARCH = 0.01

# Budget ceiling for Part 2. At $0.01/call this allows 20 iterations.
BUDGET_LIMIT = 0.20

# ── Sandbox warmup ─────────────────────────────────────────────────────────────
import uuid
from typing import Annotated, Optional, TypedDict
from figuard import FiGuardClient

figuard = FiGuardClient(api_key=FIGUARD_API_KEY, base_url=FIGUARD_BASE_URL)
print("Connecting to FiGuard sandbox…", end=" ", flush=True)
run_id = uuid.uuid4().hex[:8]
budget = figuard.create_budget(
    user_id="research_agent",
    total_limit=BUDGET_LIMIT,
    currency="USD",
    expires_in="1h",
)
print("ready.")
print(f"\nMode            : {MODE}")
print(f"Simulated iters : {SIMULATED_ITERATIONS}  (agent keeps searching until stopped)")
print(f"Cost per search : ${COST_PER_SEARCH:.3f}")
print(f"Budget ceiling  : ${BUDGET_LIMIT:.2f}  ({int(BUDGET_LIMIT / COST_PER_SEARCH)} searches)")
print(f"Dashboard       : {FIGUARD_BASE_URL}/ui")

## Part 1 — Without FiGuard

The LLM controls the loop. It never decides it has enough.
The graph runs all 30 iterations with no cost ceiling.

In [ ]:
class BudgetExhausted(Exception):
    """Raised by the search tool when FiGuard denies the authorization."""
    pass


class SearchClient:
    """Real Serper search or a simulator that returns canned results."""

    def __init__(self, mode: str, api_key: Optional[str] = None):
        self.mode = mode
        self.call_count = 0
        if mode == "real":
            import requests
            self._requests = requests
            self._api_key  = api_key

    def search(self, query: str) -> str:
        self.call_count += 1
        if self.mode == "real":
            resp = self._requests.post(
                "https://google.serper.dev/search",
                headers={"X-API-KEY": self._api_key, "Content-Type": "application/json"},
                json={"q": query},
                timeout=10,
            )
            resp.raise_for_status()
            snippets = [r.get("snippet", "") for r in resp.json().get("organic", [])[:3]]
            return " | ".join(snippets) or "No results."
        else:
            # Simulated: always returns a plausible-but-inconclusive result
            # so the agent keeps searching (realistic for ambiguous queries).
            return (
                f"[sim result {self.call_count}] Found partial information about '{query}'. "
                f"Several sources conflict. Further research recommended."
            )

    def reset(self): self.call_count = 0


def run_simulated_loop(search_client, figuard=None, token=None, label=""):
    """Runs a simulated research loop. Returns (iterations_run, total_cost)."""
    total_cost = 0.0
    query = "impact of AI agents on enterprise software spending 2024"

    for i in range(1, SIMULATED_ITERATIONS + 1):
        if figuard and token:
            auth = figuard.authorize(
                session_token=token,
                agent_id="research_agent",
                action_type="SEARCH",
                description=f"Web search iteration {i}: {query[:60]}",
                requested_quantity=COST_PER_SEARCH,
                idempotency_key=f"{label}-search-iter-{i}",
            )
            if not auth.is_authorized:
                bad(f"[FiGuard] Search {i} denied — {auth.denial_reason}")
                info(f"Agent exits loop with results from {i - 1} searches.")
                return i - 1, total_cost

            step(f"Search {i:>2}: authorized ${COST_PER_SEARCH:.2f} (event {auth.event_id[:8]}…)")
        else:
            step(f"Search {i:>2}: calling search API…")

        result = search_client.search(query)
        total_cost += COST_PER_SEARCH

        if figuard and token:
            figuard.confirm_event(auth.event_id, confirmed_quantity=COST_PER_SEARCH)

        info(f"           result: {result[:80]}…")
        info(f"           LLM: insufficient — keep searching")

    return SIMULATED_ITERATIONS, total_cost


search = SearchClient(mode=MODE)

section("PART 1 — Without FiGuard")
info(f"Query: 'impact of AI agents on enterprise software spending 2024'")
info(f"The LLM controls the loop. It never decides it has enough.")
info(f"The graph runs all {SIMULATED_ITERATIONS} iterations.\n")

search.reset()

if MODE == "simulation":
    iterations, cost = run_simulated_loop(search_client=search)
else:
    import os
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY

    from langchain_openai import ChatOpenAI
    from langchain_core.messages import HumanMessage, AIMessage, ToolMessage
    from langchain_core.tools import tool
    from langgraph.graph import StateGraph, END

    class AgentState(TypedDict):
        messages: Annotated[list, lambda x, y: x + y]
        search_count: int
        done: bool

    iteration_counter = {"n": 0}

    @tool
    def web_search_unsafe(query: str) -> str:
        """Search the web for information on a topic."""
        iteration_counter["n"] += 1
        return search.search(query)

    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_with_tools = llm.bind_tools([web_search_unsafe])

    def agent_node(state):
        response = llm_with_tools.invoke(state["messages"])
        return {"messages": [response], "search_count": state["search_count"], "done": False}

    def tool_node(state):
        last = state["messages"][-1]
        results = []
        for call in last.tool_calls:
            result = web_search_unsafe.invoke(call["args"])
            results.append(ToolMessage(content=result, tool_call_id=call["id"]))
        return {"messages": results, "search_count": state["search_count"] + len(results), "done": False}

    def should_continue(state):
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return END

    graph = StateGraph(AgentState)
    graph.add_node("agent", agent_node)
    graph.add_node("tools", tool_node)
    graph.set_entry_point("agent")
    graph.add_conditional_edges("agent", should_continue, {"tools": "tools", END: END})
    graph.add_edge("tools", "agent")
    compiled = graph.compile()

    compiled.invoke({
        "messages": [HumanMessage(content=(
            "Research the impact of AI agents on enterprise software spending in 2024. "
            "Keep searching until you have comprehensive coverage from multiple angles."
        ))],
        "search_count": 0,
        "done": False,
    })
    iterations = iteration_counter["n"]
    cost = iterations * COST_PER_SEARCH

print()
bad(f"{iterations} search calls — ${cost:.2f} spent")
bad(f"No ceiling. Cost grows with query complexity, not your budget.")

## Part 2 — With FiGuard

Same query, same graph. A $0.20 budget for this research session.
FiGuard authorizes each search call. The loop exits when the budget is hit —
the graph returns what it gathered so far.

In [ ]:
section("PART 2 — With FiGuard")
info(f"Same query, same graph. ${BUDGET_LIMIT:.2f} budget for this research session.")
info(f"FiGuard authorizes each search call. Loop exits when budget is hit.\n")

search.reset()
token = budget.session_token
ok(f"Budget: {budget.id}  (limit: ${BUDGET_LIMIT:.2f})")
info("")

if MODE == "simulation":
    iterations, cost = run_simulated_loop(
        search_client=search,
        figuard=figuard,
        token=token,
        label=run_id,
    )
else:
    iteration_counter["n"] = 0

    @tool
    def web_search_safe(query: str) -> str:
        """Search the web for information on a topic."""
        iteration_counter["n"] += 1
        i = iteration_counter["n"]

        auth = figuard.authorize(
            session_token=token,
            agent_id="research_agent",
            action_type="SEARCH",
            description=f"Web search iteration {i}: {query[:60]}",
            requested_quantity=COST_PER_SEARCH,
            idempotency_key=f"{run_id}-search-iter-{i}",
        )
        if not auth.is_authorized:
            raise BudgetExhausted(
                f"Search budget exhausted after {i - 1} searches "
                f"(${(i - 1) * COST_PER_SEARCH:.2f} spent). "
                f"Reason: {auth.denial_reason}"
            )
        step(f"Search {i:>2}: authorized (event {auth.event_id[:8]}…)")
        result = search.search(query)
        figuard.confirm_event(auth.event_id, confirmed_quantity=COST_PER_SEARCH)
        return result

    llm_safe = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    llm_with_tools_safe = llm_safe.bind_tools([web_search_safe])

    def agent_node_safe(state):
        response = llm_with_tools_safe.invoke(state["messages"])
        return {"messages": [response], "search_count": state["search_count"], "done": False}

    def tool_node_safe(state):
        last = state["messages"][-1]
        for call in last.tool_calls:
            try:
                result = web_search_safe.invoke(call["args"])
            except BudgetExhausted as e:
                result = f"BUDGET_EXHAUSTED: {e}"
                return {"messages": [ToolMessage(content=result, tool_call_id=call["id"])],
                        "search_count": state["search_count"] + 1, "done": True}
            return {"messages": [ToolMessage(content=result, tool_call_id=call["id"])],
                    "search_count": state["search_count"] + 1, "done": False}
        return {"messages": [], "search_count": state["search_count"], "done": False}

    def should_continue_safe(state):
        if state.get("done"):
            return END
        last = state["messages"][-1]
        if hasattr(last, "tool_calls") and last.tool_calls:
            return "tools"
        return END

    graph_safe = StateGraph(AgentState)
    graph_safe.add_node("agent", agent_node_safe)
    graph_safe.add_node("tools", tool_node_safe)
    graph_safe.set_entry_point("agent")
    graph_safe.add_conditional_edges("agent", should_continue_safe, {"tools": "tools", END: END})
    graph_safe.add_edge("tools", "agent")
    compiled_safe = graph_safe.compile()

    compiled_safe.invoke({
        "messages": [HumanMessage(content=(
            "Research the impact of AI agents on enterprise software spending in 2024. "
            "Keep searching until you have comprehensive coverage from multiple angles."
        ))],
        "search_count": 0,
        "done": False,
    })
    iterations = iteration_counter["n"]
    cost = iterations * COST_PER_SEARCH

print()
ok(f"{iterations} search calls — ${cost:.2f} spent")
ok(f"Loop stopped at budget ceiling. Graph exited with partial results.")
info(f"Without FiGuard: {SIMULATED_ITERATIONS} calls, "
     f"${SIMULATED_ITERATIONS * COST_PER_SEARCH:.2f}. "
     f"Saving: ${(SIMULATED_ITERATIONS - iterations) * COST_PER_SEARCH:.2f}")

## What FiGuard Recorded

Open the dashboard and find the budget printed above.
Go to **Budget → Ledger** to see every authorized and denied search event.
Go to **Overview** to see the spend bar at 100% with status EXHAUSTED.

In [ ]:
b = figuard.get_budget(budget.id)

section("What FiGuard recorded")
info(f"Dashboard : {FIGUARD_BASE_URL}/ui")
info(f"Budget    : {budget.id}")
info("")
ok(f"Spent     : ${b.quantity_spent:.2f} of ${BUDGET_LIMIT:.2f}")
ok(f"Remaining : ${b.available_quantity:.2f}")
info("")
info("Open the budget → Ledger. You'll see one CONFIRMED event per")
info("search call, then one final DENIED / INSUFFICIENT_FUNDS.")
info("That denial is where the loop stopped.")
info("")
info("Open Overview → the spend bar is at 100%. Status: EXHAUSTED.")
info("No search call slipped through after the ceiling was hit.")
print(f"\n  Dashboard link: {FIGUARD_BASE_URL}/ui")